# Sentiment Analysis Pipeline

## Project Information
- **Source**: Transformers Packt Course (lazyprogrammer.me/course_files/nlp/)
- **Objective**: Perform sentiment analysis on text data using Hugging Face Transformers
- **Pipeline**: `sentiment-analysis`
- **Dataset**: Airline Tweets (Tweets.csv)

## Overview
This notebook demonstrates how to use the Hugging Face Transformers sentiment-analysis pipeline to classify text as positive or negative sentiment.


In [ ]:
%pip install transformers pandas numpy matplotlib seaborn scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')


## Initialize Sentiment Analysis Pipeline


In [ ]:
# Initialize the sentiment analysis pipeline
classifier = pipeline("sentiment-analysis")
print(f"Pipeline type: {type(classifier)}")

# Test with simple examples
test_texts = ["This is such a great movie!", "I don't understand"]
results = classifier(test_texts)
for text, result in zip(test_texts, results):
    print(f"Text: {text}")
    print(f"Label: {result['label']}, Score: {result['score']:.4f}\n")


## Load and Prepare Dataset


In [ ]:
# Load airline tweets dataset
df = pd.read_csv('Tweets.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()


In [ ]:
# Prepare data: keep only sentiment and text, remove neutral
df = df[['airline_sentiment', 'text']]
df = df[df['airline_sentiment'] != "neutral"]

# Map labels to binary (positive=1, negative=0)
label_map = {'positive': 1, 'negative': 0}
df['target'] = df['airline_sentiment'].map(label_map)

print(f"Dataset shape after filtering: {df.shape}")
print(f"\nLabel distribution:")
print(df['airline_sentiment'].value_counts())
df.head()


## Evaluate on Sample Data


In [ ]:
# Sample a small subset for evaluation
sample_size = 100
sdf = df.sample(n=sample_size, random_state=42)
print(f"Sample size: {len(sdf)}")

# Get predictions
predictions = classifier(sdf['text'].to_list())
print(f"\nFirst prediction example:")
print(predictions[0])


In [ ]:
# Extract predictions and probabilities
pred_labels = [pred['label'] for pred in predictions]
sent_map = {'NEGATIVE': 0, 'POSITIVE': 1}
preds_mapped = [sent_map[label] for label in pred_labels]

# Calculate probabilities (convert classes to scores for positive class)
probs = [pred['score'] if pred['label'].startswith('P') else 1 - pred['score'] 
         for pred in predictions]

# Convert to numpy array
preds = np.array(preds_mapped)

print(f"Predictions shape: {preds.shape}")
print(f"True labels shape: {sdf['target'].values.shape}")


## Model Evaluation


In [ ]:
# Calculate metrics
accuracy = accuracy_score(sdf['target'], preds)
f1 = f1_score(sdf['target'], preds)

print("Model Performance:")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(sdf['target'], preds)
print(f"\nConfusion Matrix:")
print(cm)


In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix - Sentiment Analysis')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()
